In [68]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("research_paper02.pdf")
document = loader.load()

In [69]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [70]:
chunks = text_splitter.split_documents(document)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5923.91it/s]


In [ ]:
from langchain_community.vectorstores import Chroma

vector_storage = Chroma.from_documents(
    documents = chunks,
    embedding = embeddings, 
    persist_directory = "dynamic_database01"
)

In [ ]:
retriever = vector_storage.as_retriever(
    search_type = "similarity",
    search_kwargs = {"k":2}
)

In [ ]:
# chat_history = []  # for storing chat history

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an helpful LLM Assistant whose job is to accurately monitor the"
    " context from given context: {context} and then give precise and accurate answers to"
    " user's questions without hillucinating important and indivisual specific data."
    "Note: If the relevant context is not found, just tell the user that we dont have what you"
    "are looking for without guessing blindly"
    # "This is what the user talked about previously and how you responded: {chat_history}"
    # " (if it is null, then it means you haven't talked about anything)"
    ),
    ("user", "Give me precis and accurate knowledge about the asked question: {question}")
])

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

output_parser = StrOutputParser()

# print(RunnablePassthrough)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY)




In [ ]:
chain = ({
    "context": retriever, 
    "question": RunnablePassthrough()
}
| prompt 
| llm
| output_parser
)

In [ ]:
def ask(question):
    response = chain.invoke(question)
    print(response)
    # chat_history.append({question,response})

In [ ]:
q01 = "How many Electromagnetic Testing Rooms do we have and what is the size of its complex?"
ask(q01)

You have 8 Electromagnetic Testing Rooms, and the size of the complex is 228,000 sq ft.


In [ ]:
# q02 = "What did we talk about just now, give me details about that topic?"
# ask(q02)

Based on the provided context, we just discussed the following details:

1.  **Core Values:**
    *   Scientific Integrity
    *   Engineering Excellence
    *   Privacy by Design
    *   Transparency
    *   Long-Term Innovation
    *   Ethical Cryptography
    *   Operational Resilience

2.  **Corporate Colors:**
    *   Midnight Blue
    *   Graphite Black
    *   Electric Cyan

3.  **Official Internal Nickname:**
    *   "The Mesh Company"

4.  **Company Background (Aethelion Systems):**
    *   **Origins:** The company's origins trace back to a research collaboration at the Massachusetts Institute of Technology (MIT) in late 2012.
    *   **Cybersecurity Context (2012):** At that time, cybersecurity discussions primarily focused on improving existing encryption standards, rather than fundamentally redesigning enterprise network architecture.
